# Köppen-Geiger como filtro de selección de donante (no como ajuste de magnitud)

**Aviso importante antes de arrancar (para no repetir el error de encuadre que casi
pasa acá):** Köppen-Geiger NO es un cuarto candidato para el mismo mecanismo de razón
que probamos con NASA POWER (Hallazgo 25), GWA (Hallazgo 26) y ERA5 (Hallazgo 27/28).
Ese mecanismo necesita una fuente CONTINUA de viento en cualquier punto -- Köppen no
da eso, da una ETIQUETA categórica de zona climática (ej. `Aw`, `Cfb`, `BSh`) derivada
de temperatura y precipitación histórica, no de viento.

**El problema que sí ataca Köppen es otro, previo:** `vecino_mas_cercano()`
(`engine/formas_regionales.py`) elige de qué estación real tomar prestada la FORMA de
la curva de excedencia usando pura distancia geográfica (Haversine). Eso puede fallar
cuando el punto más cercano en línea recta está en un régimen climático distinto del
punto exacto (ladera de barlovento vs. sotavento, costa vs. interior, etc.) -- exactamente
el tipo de cosa que la app necesita manejar bien por ser internacional (Hallazgo 24) y
que probablemente sea parte de por qué el ráster crudo de GWA se aleja tanto de la
realidad en algunos sitios (Hallazgo 26). Köppen puede servir como un FILTRO o
DESEMPATE en esa selección: preferir un donante en la misma zona climática (o una
climatológicamente parecida) aunque esté un poco más lejos, en vez de tomar siempre
el más cercano en distancia pura.

Esto es complementario al trabajo de ajuste de magnitud (NASA POWER/GWA/ERA5), no un
sustituto -- atacan dos partes distintas del mismo problema (¿de quién copio la FORMA?
vs. ¿cuánto reescalo la MAGNITUD?).

### Por qué esta fuente en particular (pedido de Pablo: no-meteorológica, no de pago)

Beck et al. 2018 (*Present and future Köppen-Geiger climate classification maps at
1-km resolution*, Scientific Data) -- no es un reanálisis meteorológico ni un servicio
con cola de procesamiento como CDS/ERA5: es un raster estático ya calculado, publicado
en Figshare, descarga directa sin registro ni token.

In [1]:
import os

def _find_repo_root():
    for candidato in ("..", "/content/ECO-Wind"):
        if os.path.exists(os.path.join(candidato, ".git")):
            return os.path.abspath(candidato)
    return None

repo = _find_repo_root()
if repo is None:
    repo = "/content/ECO-Wind"
    get_ipython().system(f"git clone https://github.com/Sogo2012/ECO-Wind.git {repo}")
else:
    get_ipython().system(f"git -C {repo} fetch origin main")
    get_ipython().system(f"git -C {repo} reset --hard origin/main")

get_ipython().run_line_magic("cd", f"{repo}/notebooks")
get_ipython().system(f"git -C {repo} log -1 --format='Commit activo: %h  %s  (%ci)'")

From https://github.com/Sogo2012/eco-wind
 * branch            main       -> FETCH_HEAD


HEAD is now at 9631bda feat(fase2): Hallazgo 27 -- acceso real confirmado a Köppen (Figshare), celda de descarga+extracción


/home/user/eco-wind/notebooks
Commit activo: 9631bda  feat(fase2): Hallazgo 27 -- acceso real confirmado a Köppen (Figshare), celda de descarga+extracción  (2026-08-31 23:58:31 +0000)


In [2]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import requests

from engine.formas_regionales import cargar_formas_conocidas, vecino_mas_cercano

pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## Parte 1 — Acceso real al dataset (sin adivinar nombres de archivo)

`figshare.com` está bloqueado en este sandbox (mismo problema que con GWA/CDS,
Hallazgo 2) -- así que, siguiendo la misma disciplina que con esas dos fuentes, esto
se corre primero en Colab para confirmar el acceso real antes de asumir nada. La API
pública de Figshare (`api.figshare.com/v2/articles/{id}`, documentada, sin auth para
artículos públicos) debería devolver el listado real de archivos del dataset -- eso
nos da los nombres/URLs reales de descarga en vez de adivinarlos.

In [3]:
try:
    resp = requests.get("https://api.figshare.com/v2/articles/6396959", timeout=20)
    resp.raise_for_status()
    datos = resp.json()
    print(f"Artículo: {datos.get('title')!r}")
    print(f"{len(datos.get('files', []))} archivo(s):")
    for f in datos.get("files", []):
        print(f"  - {f['name']}  ({f['size'] / 1e6:.1f} MB)  {f['download_url']}")
except Exception as exc:
    print(f"FALLO: {exc!r}")

Artículo: 'Present and future Köppen-Geiger climate classification maps at 1-km resolution'
1 archivo(s):
  - Beck_KG_V1.zip  (71.0 MB)  https://ndownloader.figshare.com/files/12407516


In [4]:
import os
import zipfile

RUTA_ZIP = "../datos_clima/koppen/Beck_KG_V1.zip"
RUTA_EXTRAIDO = "../datos_clima/koppen/extraido"

try:
    os.makedirs(os.path.dirname(RUTA_ZIP), exist_ok=True)
    if not os.path.exists(RUTA_ZIP):
        resp = requests.get("https://ndownloader.figshare.com/files/12407516", stream=True, timeout=180)
        resp.raise_for_status()
        with open(RUTA_ZIP, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1 << 20):
                f.write(chunk)
        print(f"Descargado: {RUTA_ZIP} ({os.path.getsize(RUTA_ZIP) / 1e6:.1f} MB)")
    else:
        print(f"Ya existía: {RUTA_ZIP}")

    with zipfile.ZipFile(RUTA_ZIP) as z:
        nombres = z.namelist()
        print(f"\n{len(nombres)} archivo(s) dentro del zip:")
        for n in nombres:
            print(f"  - {n}")
        z.extractall(RUTA_EXTRAIDO)
    print(f"\nExtraído en: {RUTA_EXTRAIDO}")

    ruta_legend = None
    for raiz, _, archivos in os.walk(RUTA_EXTRAIDO):
        for a in archivos:
            if a.lower() == "legend.txt":
                ruta_legend = os.path.join(raiz, a)
    if ruta_legend:
        print(f"\n--- {ruta_legend} ---")
        print(open(ruta_legend).read())
    else:
        print("\nNo se encontró legend.txt -- revisar el listado de arriba a mano.")
except Exception as exc:
    print(f"FALLO: {exc!r}")

FALLO: ProxyError(MaxRetryError("HTTPSConnectionPool(host='ndownloader.figshare.com', port=443): Max retries exceeded with url: /files/12407516 (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))



### Nombre real del archivo (confirmado por Pablo, no adivinado)

Pablo bajó y descomprimió el zip a mano y mandó el listado real. Dentro hay 16 archivos:
presente/futuro × 3 resoluciones (`0p5` ≈ 50km, `0p083` ≈ 9km, `0p0083` ≈ 1km) × mapa de
clasificación vs. mapa de confianza (`_conf_`), más el `legend.txt` (confirmado -- 30 clases,
igual al legend.txt oficial del paper) y un archivo suelto `KoppenGeiger` (fuente Objective-C,
no es parte del dataset, se ignora).

El que necesitamos: **`Beck_KG_V1_present_0p0083.tif`** -- presente (no `future_`), clasificación
(no `_conf_`, que es la capa de confianza/incertidumbre), la resolución más fina (0.0083° ≈ 1km).


## Parte 2 — Mecanismo propuesto (boceto, todavía no integrado a `engine/`)

Extender `vecino_mas_cercano()` para que, dado un raster de zonas Köppen ya
descargado, filtre o repondere los candidatos por coincidencia de zona antes de
elegir por distancia -- por ejemplo: quedarse solo con donantes en la misma zona (o
una zona vecina en el esquema de Köppen-Geiger) cuando exista al menos uno dentro de
un radio razonable, y caer de nuevo a pura distancia si no hay ninguno.

Todavía no se decide la regla exacta de desempate (¿misma zona exacta? ¿mismo grupo
principal -- A/B/C/D/E -- si no hay match exacto cerca?) ni se validó si esto de
verdad mejora el leave-one-out de Hallazgo 21/22 frente a la distancia pura -- eso es
el próximo paso, una vez confirmado el acceso real al raster (Parte 1).

In [5]:
def muestrear_zona_koppen(lat, lon, ruta_raster="../datos_clima/koppen/extraido/Beck_KG_V1_present_0p0083.tif"):
    """
    Mismo patrón que muestrear_velocidad_media() en engine/gwa_raster.py --
    import perezoso de rasterio, no se ejecuta acá (sandbox). Ruta por defecto
    ya apunta al archivo real confirmado (ver celda de arriba), presente + 1km
    + clasificación (no el mapa de confianza).
    """
    import rasterio

    with rasterio.open(ruta_raster) as src:
        fila, columna = src.index(lon, lat)
        banda = src.read(1)
        return int(banda[fila, columna])


LEYENDA_KOPPEN = {
    1: "Af", 2: "Am", 3: "Aw", 4: "BWh", 5: "BWk", 6: "BSh", 7: "BSk",
    8: "Csa", 9: "Csb", 10: "Csc", 11: "Cwa", 12: "Cwb", 13: "Cwc",
    14: "Cfa", 15: "Cfb", 16: "Cfc", 17: "Dsa", 18: "Dsb", 19: "Dsc", 20: "Dsd",
    21: "Dwa", 22: "Dwb", 23: "Dwc", 24: "Dwd", 25: "Dfa", 26: "Dfb", 27: "Dfc",
    28: "Dfd", 29: "ET", 30: "EF",
}  # de legend.txt, confirmado por Pablo -- no adivinado


def vecino_mas_cercano_por_zona(lat, lon, formas, ruta_raster_koppen, excluir=None, radio_km=150):
    """
    Boceto -- todavía no reemplaza a vecino_mas_cercano() en engine/formas_regionales.py.
    Idea: entre los donantes dentro de radio_km, preferir los que comparten zona
    Köppen con el punto exacto; si ninguno la comparte, caer a distancia pura
    (mismo comportamiento que hoy).
    """
    zona_exacta = muestrear_zona_koppen(lat, lon, ruta_raster_koppen)

    candidatos = []
    for clave, sitio in formas.items():
        if clave == excluir:
            continue
        zona_donante = muestrear_zona_koppen(sitio["lat"], sitio["lon"], ruta_raster_koppen)
        candidatos.append((clave, zona_donante == zona_exacta))

    # TODO: combinar con la distancia real (vecino_mas_cercano ya calculada) --
    # esto es solo el filtro de zona, falta la lógica de desempate/radio.
    return candidatos


### Prueba rápida -- sanity check (correr después de la celda de descarga, Parte 1)

Antes de meterse con la regla de desempate: ¿la zona que devuelve el raster para los 4 sitios
conocidos suena razonable? San José/Valle Central debería salir templado (`Cwb`/`Cfb`-ish,
tierras altas tropicales), Guanacaste (Nicoya/Liberia) debería salir tropical seco (`Aw`/`BSh`),
Limón/Caribe debería salir tropical húmedo (`Af`/`Am`). Si algo de esto sale claramente
distinto, hay un problema antes de construir nada más encima.


In [6]:
formas = cargar_formas_conocidas()

try:
    for clave, sitio in formas.items():
        codigo = muestrear_zona_koppen(sitio["lat"], sitio["lon"])
        print(f"{sitio['nombre']:55s} -> {codigo:2d}  {LEYENDA_KOPPEN.get(codigo, '?')}")
except Exception as exc:
    print(f"FALLO: {exc!r}")

FALLO: RasterioIOError('../datos_clima/koppen/extraido/Beck_KG_V1_present_0p0083.tif: No such file or directory')


## Estado honesto

- Parte 1 (acceso al dataset): **sin correr todavía** -- pendiente de ejecutar esto en
  Colab (igual que se hizo con GWA/ERA5) antes de asumir que la URL/API responde como
  se espera.
- Parte 2 (mecanismo): boceto sin terminar ni validar -- falta la regla de desempate,
  la integración real con `vecino_mas_cercano()`, y una validación leave-one-out
  (mismo patrón de Hallazgo 21/22) comparando contra la selección por distancia pura.
- No conectado a `app.py`. No reemplaza ni compite con el trabajo de ajuste de
  magnitud (NASA POWER/GWA/ERA5) -- es un eje distinto del mismo problema más grande.